# Language Table: paper Tables 1 and 2 from raw per-episode results

This notebook recomputes the two Language Table tables of the WMM paper:

| # | Paper table | What it reports |
|---:|---|---|
| 1 | Tab. 1 *Language Table* (`tab:exp:lt`) | Success rate (%) of each method on the five task families, plus their plain average |
| 2 | Tab. 2 *Plain Policy versus MPC* (`tab:exp:lt:mpc`) | Success rate and number of steps to success of the same WMM checkpoint used as a plain policy and inside model predictive control (MPC), on B2B and B2AL |

It reads only the per-episode files under `release_data/langtable/L1_results/` (or `$WMM_DATA/langtable/L1_results/` when the environment variable `WMM_DATA` is set). It does not run the simulator or the model.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from lt_tables import (COLUMNS, DIGITS, FAMILIES, PAPER_TABLE1, PAPER_TABLE2, QUOTED_FROM, STATS,
                       check_results_json, find_repo_root, langtable_data, load_episodes,
                       policy_stats, success_rates)

REPO_ROOT = find_repo_root(Path.cwd().resolve())
L1 = langtable_data(REPO_ROOT) / 'L1_results'
if not L1.is_dir():
    raise FileNotFoundError(f'{L1} not found: get the Language Table L1 results (apps/langtable/README.md) or set WMM_DATA')
print('reading', L1.relative_to(REPO_ROOT) if L1.is_relative_to(REPO_ROOT) else L1)

## 1. Load every evaluation episode

The benchmark has five task families. Each family is one *reward factory* of the Language Table simulator, i.e. one rule that decides when an instruction counts as done.

| # | Column | Folder | Example instruction |
|---:|---|---|---|
| 1 | B2B | `blocktoblock` | put the green star close to the green cube |
| 2 | B2AL | `blocktoabsolutelocation` | slide the blue cube to the center |
| 3 | B2BRL | `blocktoblockrelativelocation` | slide the green star to the right of the green cube |
| 4 | B2RL | `blocktorelativelocation` | slide the blue cube upwards and then to the right |
| 5 | Sep | `separate` | separate the red moon from the blue moon |

Each family was evaluated with 50 episodes (rollouts) of at most 200 simulator steps. One episode is one small JSON file:

| # | Run | File | Fields used here |
|---:|---|---|---|
| 1 | Plain policy | `plain/<folder>/<NNNN>/meta.json` | `ep_num`, `succ`, `succ_steps`, `instruction` |
| 2 | MPC | `mpc/<folder>/ep_<N>/meta.json` | same fields |

`succ` is true when the family's success rule fired before step 200. `succ_steps` is the index (counting from 0) of the simulator step on which it fired; it is -1 for a failed episode.

In [ ]:
episodes = pd.concat([load_episodes(L1 / 'plain', 'plain'), load_episodes(L1 / 'mpc', 'mpc')], ignore_index=True)
print(episodes.groupby(['mode', 'family']).size().unstack()[list(FAMILIES)])
episodes.head()

Each evaluation run also wrote its own summary, `results.json`. The next cell checks that the per-episode files and that summary agree episode by episode, so the numbers below do not depend on which of the two you trust.

In [ ]:
for mode in ['plain', 'mpc']:
    check_results_json(episodes[episodes['mode'] == mode], L1 / mode, mode)
print('per-episode files agree with results.json of both runs (success and steps of all 500 episodes)')

## 2. Table 1: success rate per task family

**Success rate** of a family = successful episodes / 50, in percent. **Avg** = the plain mean of the five family success rates (all families have 50 episodes, so this also equals all successes / 250).

Only the WMM row is recomputed here: it comes from the plain-policy run above. Every other row is quoted from the paper that reported it:

| # | Rows | Source |
|---:|---|---|
| 1 | Scratch, UniPi, VPT, LAPA, ActionVLA | LAPA paper (Ye et al., arXiv 2410.11758), table *Language Table In-Domain Seen Results* |
| 2 | LCB, LangTable orig. | LCB paper (Shentu et al., arXiv 2405.04798), Table I; *LangTable orig.* is LCB's re-run of the public Lynch et al. 2022 checkpoint |

In [ ]:
plain = episodes[episodes['mode'] == 'plain']
counts = plain.groupby('family').success.agg(successes='sum', episodes='count').loc[list(FAMILIES)]
print(counts.T.to_string(), end='\n\n')

ours = success_rates(plain)

table1 = pd.DataFrame(
    {m: (v if m in QUOTED_FROM else [ours[c] for c in COLUMNS]) for m, v in PAPER_TABLE1.items()},
    index=COLUMNS).T
print('Table 1 (success rate %, WMM row recomputed, other rows quoted):')
print(table1.to_string(float_format=lambda x: f'{x:.1f}'))

## 3. Table 2: plain policy versus MPC

The same WMM checkpoint is run in two ways on the two families whose instruction names an absolute target (B2B, B2AL):

| # | Variant | What drives the robot |
|---:|---|---|
| 1 | Plain | The model predicts the next 10 actions from the last 5 observed steps and the instruction; they are executed as predicted |
| 2 | MPC | Before executing, 4 candidate 10-step action chunks are optimized by gradient descent through the model so that the predicted final block positions approach the goal the model imagined; the best one is executed |

Rows of the table:

| # | Row | Meaning |
|---:|---|---|
| 1 | Succ. | Success rate (%) over the 50 episodes |
| 2 | Mean / Med. / Min / Max steps | Mean, median, minimum and maximum of `succ_steps`, taken over the **successful episodes only** (a failed episode is left out, not counted as 200). Fewer steps means the task was solved faster |

The plain columns are the same plain-policy run as the WMM row of Table 1.

In [ ]:
table2 = pd.DataFrame(
    {(fam, variant): policy_stats(episodes[(episodes['mode'] == variant.lower()) & (episodes.family == fam)])
     for fam, variant in PAPER_TABLE2},
    index=STATS)
print('Table 2 (recomputed):')
print(table2.to_string(float_format=lambda x: f'{x:.2f}'))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, fam in zip(axes, ['B2B', 'B2AL']):
    data, labels = [], []
    for variant in ['Plain', 'MPC']:
        df = episodes[(episodes['mode'] == variant.lower()) & (episodes.family == fam)]
        data.append(df[df.success].steps)
        labels.append(f'{variant}\n{int(df.success.sum())}/50 succeeded')
    ax.boxplot(data, vert=False, showmeans=True)
    ax.set_yticks([1, 2], labels)
    ax.set_title(fam)
    ax.set_xlabel('steps to success (successful episodes)')
fig.tight_layout()
plt.show()